# PA2 runbook
Run on a Colab GPU runtime. Each stage writes into `results/`, `figures/`, `gifs/` and `checkpoints/`. Two commit points are marked. Do not run the cells after a commit point before the commit is pushed.

## 0. Setup

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
STUDENT_NAME = 'YOUR FULL NAME'
ROLL_NUMBER = 'YOUR ROLL NUMBER'
PUBLIC_URL = 'https://github.com/YOUR_NAME/YOUR_REPO'  # goes on page 1 of the report, the repo must be public at submission
REPO_URL = PUBLIC_URL.replace('https://', 'https://' + userdata.get('GH_TOKEN') + '@') + '.git'  # token comes from Colab Secrets, never type it here
%cd /content/drive/MyDrive
!git clone $REPO_URL pa2 || git -C pa2 pull
%cd /content/drive/MyDrive/pa2
!pip -q install opencv-python-headless imageio latex2mathml markdown playwright pypandoc_binary
!playwright install --with-deps chromium
!git config --global user.email 'YOUR_EMAIL' && git config --global user.name 'YOUR_NAME'
!python -c "import torch;print(torch.__version__, torch.cuda.is_available())"


In [ ]:
!python tests/test_all.py

## 1. Setup checks, Figure 1 and reach
Outputs Table 1, Figure 1 and the reach numbers.

In [ ]:
!python run.py table1
!python run.py flow
!python run.py reach

## 2. Baseline model A
Trains A with seeds 0 and 1, then prints the 1x baseline numbers. No mirror, rotation or speed test is run here.

In [ ]:
import os
for s in [0, 1]:
    if not os.path.exists(f'checkpoints/A_s{s}.pt'):
        !python run.py train --model A --seed $s
!python run.py table2
!cat results/table2_preview.json


### COMMIT POINT 1
Generate the Q2.5 draft from Figure 1 and the reach. Read it, edit it if it does not match your own reasoning, then commit and push it before running the next stage.

In [ ]:
!python predictions/make_q25.py

In [ ]:
!git add predictions/Q2.5.md && git commit -m 'Q2.5 prediction before testing model A' && git push

## 3. Testing the baseline
Mirror, rotation and speed tests for model A. Figures 2 to 5 use model A only.

In [ ]:
!python run.py evaluate --models A
!python run.py select_videos
!python run.py figures

### COMMIT POINT 2
Generate the Q2.11 draft from the failures of A. Edit if needed, then commit and push before training B, C or D. Candidate designs for D count as training D, so they also come after this commit.

In [ ]:
!python predictions/make_q211.py

In [ ]:
!git add predictions/Q2.11.md && git commit -m 'Q2.11 prediction before training B, C and D' && git push

## 4. Models B, C and D
Select the design of D on the validation split, then train B, C and D with both seeds. Lower `--epochs` in the selection step to save time and disclose it.

In [ ]:
!python run.py select_d --epochs 10
!cat results/d_selection.json

In [ ]:
import os
for m in ['B', 'C', 'D']:
    for s in [0, 1]:
        if not os.path.exists(f'checkpoints/{m}_s{s}.pt'):
            !python run.py train --model $m --seed $s


Optional model D+B. Train it only if Table 5 shows that symmetry augmentation helps D, then use `--final DB` in the last two stages.

In [ ]:
# for s in [0, 1]:
#     !python run.py train --model DB --seed $s

## 5. Evaluation, final figures and report

In [ ]:
!python run.py evaluate --models A B C D
!python run.py select_videos
!python run.py figures --final D
!python run.py diagnostics --final D

In [ ]:
!python report/make_report.py --q2 --final D --repo {PUBLIC_URL} --name "{STUDENT_NAME}" --roll "{ROLL_NUMBER}"

The report is written to `report/PA2_report.pdf` and `report/PA2_report.docx`. The script exits with a nonzero status and lists any pending item. Check Figure 2, Figure 5 and the video of Q2.7 by eye, because the video rules use a blob count proxy for digit overlap. Write the AI use record in the last section of the docx, then export the docx to PDF.


In [ ]:
!git add -A && git commit -m 'Results, figures, GIFs and report' && git push
